In [44]:
import numpy as np
import rdkit
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem

In [2]:
def hasKMemberAromaticRing(mol,K=6,min_C=0.5):
    min_C=round(K*min_C)
    c=0
    for a in mol.GetAromaticAtoms():
        if not a.IsInRingSize(K): continue
        if a.GetAtomicNum()==6: # Carbon
            c+=1
    return c>=min_C
def hasKMemberAliphaticRing(mol,K=6,min_C=0.5):
    min_C=round(K*min_C)
    c=0
    for a in mol.GetAtoms():
        if not a.IsInRingSize(K): continue
        if a.GetIsAromatic(): continue # Aliphatic only
        if a.GetAtomicNum()==6: # Carbon
            c+=1
    return c>=min_C
def loadLigandCoordinates(cif_path):
    try:
        readcif=open(cif_path,"r")
    except:
        raise ValueError("File read error!")
    coords=[]
    for l in readcif:
        l=l.strip()
        if l[:6]=="HETATM":
            l=l.split()
            coords.append([float(l[10]),float(l[11]),float(l[12])])
    coords=np.array(coords)
    return coords
def checkPlanarity(coords,chk_cutoff=1.0):
    v1=coords[1]-coords[0]
    v2=coords[2]-coords[1]
    n_hat=np.cross(v1,v2)
    n_hat/=np.linalg.norm(n_hat)
    chk=0
    for i in range(3,len(coords)):
        v_test=coords[i]-coords[0]
        cos_ang=np.sum(v_test*n_hat)/np.linalg.norm(v_test)
        chk+=np.abs(cos_ang)
    return chk,(chk<chk_cutoff)

In [36]:
def _expand(base_bond,rbonds,end_points=None,curseq=None,verbose=False):
    if curseq is None: curseq=[]
    final_seq=curseq
    #print(base_bond.GetBeginAtomIdx(),"-",base_bond.GetEndAtomIdx())
    if end_points is None:
        curseq.append(base_bond)
        end_points=(base_bond.GetBeginAtomIdx(),base_bond.GetEndAtomIdx())
    if verbose: print("\t"*(len(curseq)-1),end_points)
    for i,b in enumerate(rbonds):
        if b in curseq: continue
        b0=b.GetBeginAtomIdx()
        b1=b.GetEndAtomIdx()
        nend=None
        if b0==end_points[0]:
            nend=(b1,end_points[1])
        elif b1==end_points[0]:
            nend=(b0,end_points[1])
        #elif b0==end_points[1]:
        #    nend=(end_points[0],b1)
        #elif b1==end_points[1]:
        #    nend=(end_points[0],b0)
        if nend is None: continue
        eseq=_expand(b,rbonds,nend,curseq+[b])
        if len(eseq)>len(final_seq): final_seq=eseq
    return final_seq
            
def num_consecutive_rotable(mol, verbose=False, get_rotables=False):
    mol=Chem.rdmolops.RemoveAllHs(mol)
    rbonds=[]
    for b in mol.GetBonds():
        if b.IsInRing(): continue # Not rotable
        if len(b.GetBeginAtom().GetBonds())==1 or len(b.GetEndAtom().GetBonds())==1: continue
        skip=False
        if b.GetBeginAtom().GetAtomicNum()!=6 and b.GetEndAtom().GetAtomicNum() in [6, 16]: # and b.GetEndAtom().GetHybridization()==Chem.rdchem.HybridizationType.SP2:
            for nhb in b.GetEndAtom().GetBonds():
                if nhb.GetOtherAtom(b.GetEndAtom()).GetIdx()==b.GetBeginAtomIdx(): continue
                if nhb.GetOtherAtom(b.GetEndAtom()).GetAtomicNum()==8:
                    if verbose: print("\t",nhb,nhb.GetBeginAtomIdx(),nhb.GetEndAtomIdx())
                    skip=True
                    break
        elif b.GetBeginAtom().GetAtomicNum() in [6,16] and b.GetEndAtom().GetAtomicNum()!=6: # and b.GetBeginAtom().GetHybridization()==Chem.rdchem.HybridizationType.SP2:
            for nhb in b.GetBeginAtom().GetBonds():
                if nhb.GetOtherAtom(b.GetBeginAtom()).GetIdx()==b.GetEndAtomIdx(): continue
                if nhb.GetOtherAtom(b.GetBeginAtom()).GetAtomicNum()==8:
                    if verbose: print("\t",nhb,nhb.GetBeginAtomIdx(),nhb.GetEndAtomIdx())
                    skip=True
                    break
        if skip: continue
        rbonds.append(b)
    if len(rbonds)>=2: nrot=1
    else: return len(rbonds)
    if verbose:
        for b in rbonds: print(b.GetBeginAtomIdx(),"-",b.GetEndAtomIdx())
        print()
    
    #expanding_ends=[(b.GetBeginAtomIdx(),b.GetEndAtomIdx(),[b]) for b in rbonds]
    expanding_ends=[]
    for b in rbonds:
        try:
            expanding_ends.append(_expand(b,rbonds,verbose=verbose))
            if verbose: print("Done!")
        except KeyboardInterrupt: break
    '''
    mod=[True for _ in expanding_ends]
    while any(mod):
        for ei,e in enumerate(expanding_ends):
            curbonds=e[-1]
            if verbose: print(curbonds[0].GetBeginAtomIdx(),"-",curbonds[0].GetEndAtomIdx(),"\t",e[0],e[1])
            mod[ei]=False
            for nb in rbonds:
                if nb in curbonds: continue
                if nb.GetBeginAtomIdx()==e[0] or nb.GetEndAtomIdx()==e[0]:
                    if verbose: print("\t",nb.GetBeginAtomIdx(),nb.GetEndAtomIdx())
                    curbonds.append(nb)
                    expanding_ends[ei]=(nb.GetEndAtomIdx() if nb.GetBeginAtomIdx()==e[0] else nb.GetBeginAtomIdx(),e[1],curbonds)
                    mod[ei]=True
                elif nb.GetBeginAtomIdx()==e[1] or nb.GetEndAtomIdx()==e[1]:
                    if verbose: print("\t",nb.GetBeginAtomIdx(),nb.GetEndAtomIdx())
                    curbonds.append(nb)
                    expanding_ends[ei]=(e[0],nb.GetEndAtomIdx() if nb.GetBeginAtomIdx()==e[1] else nb.GetBeginAtomIdx(),curbonds)
                    mod[ei]=True
                else: continue
                if verbose: print("\t",nb.GetBeginAtomIdx(),"-",nb.GetEndAtomIdx())
            if verbose: print()
            if mod[ei]: break
    '''
    #print(expanding_ends)
    ret=max([len(e) for e in expanding_ends])
    if get_rotables: return ret,rbonds,expanding_ends
    else: return ret

In [48]:
from IPython.display import SVG

'''
#mol=Chem.MolFromSmiles("CCCCCCCCOCCCN")
#mol=Chem.MolFromSmiles("CN([CH](Cc1ccc(cc1)c2ccccc2)C(=O)N[CH](Cc3c[nH]c4ccccc34)C(O)=O)C(=O)c5cc(C)cc(C)c5")
#mol=Chem.MolFromSmiles("COc1ccccc1Oc2c(N[S](=O)(=O)c3ccc(cc3)C(C)(C)C)nc(nc2OCCC(=O)Nc4c(C)cccc4C)c5ncccn5")
#mol=Chem.MolFromSmiles("CCCCCCCCCCCCC#CC#CCCCCC#CC#CCCCCCCCCC(=O)O")
ncrot,rbonds,ee=num_consecutive_rotable(mol,get_rotables=True,verbose=False)
print("Total N-rot:",AllChem.CalcNumRotatableBonds(mol))
print("Consec. N-rot:",ncrot)

eres=ee[np.argmax([len(e) for e in ee])]
hats=[]
for b in eres:
    hats.append(b.GetBeginAtomIdx())
    hats.append(b.GetEndAtomIdx())
hats=set(hats)
hats=list(hats)
hats

d2d = Draw.MolDraw2DSVG(550,350)
d2d.DrawMolecule(mol,highlightAtoms=hats)
d2d.FinishDrawing()
SVG(d2d.GetDrawingText())
'''
print("Drawing Skipped. Molecule Chemistry loaded successfully")

Drawing Skipped. Molecule Chemistry loaded successfully
